In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType
import pyspark.sql.functions as F
from pyspark.sql.window import Window
import os

# Spark NO soporta Java 25 (el que suele estar en el PATH). Fijamos Java 21.
JAVA21 = r"C:\Program Files\Java\jdk-21.0.10"
if os.path.isdir(JAVA21):
    os.environ["JAVA_HOME"] = JAVA21
    print(f"JAVA_HOME -> {JAVA21}")
else:
    print(f"AVISO: no existe {JAVA21}. Se usara el java del PATH. "
          f"JAVA_HOME actual: {os.environ.get('JAVA_HOME', '(sin definir)')}")

spark = SparkSession.builder \
    .appName("DeteccionFraude_PaySim") \
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()

print("Sesión de Spark iniciada con éxito.")

In [ ]:
# Se define el esquema del dataset, para facilitar el trabajo a PySpark
schema = StructType([
    StructField("step", IntegerType(), True),
    StructField("type", StringType(), True),
    StructField("amount", DoubleType(), True),
    StructField("nameOrig", StringType(), True),
    StructField("oldbalanceOrg", DoubleType(), True),
    StructField("newbalanceOrig", DoubleType(), True),
    StructField("nameDest", StringType(), True),
    StructField("oldbalanceDest", DoubleType(), True),
    StructField("newbalanceDest", DoubleType(), True),
    StructField("isFraud", IntegerType(), True),
    StructField("isFlaggedFraud", IntegerType(), True)
])

# --- Localizar el dataset -------------------------------------------------
# El CSV de PaySim pesa ~470 MB y NO se versiona. Descargalo de Kaggle:
#   https://www.kaggle.com/datasets/ealaxi/paysim1
# y colocalo junto a este notebook. Se prueban varias ubicaciones por comodidad.
NOMBRE_CSV = "PS_20174392719_1491204439457_log.csv"
CANDIDATOS = [
    NOMBRE_CSV,                                                        # junto al notebook
    os.path.join("data", NOMBRE_CSV),                                  # subcarpeta data/
    os.path.expanduser(r"~/Documents/Datacamp PySpark/" + NOMBRE_CSV),  # ubicacion original
]
path = next((p for p in CANDIDATOS if os.path.exists(p)), None)

if path is None:
    raise FileNotFoundError(
        "No se encontro el dataset PaySim.\n"
        f"Descarga '{NOMBRE_CSV}' de https://www.kaggle.com/datasets/ealaxi/paysim1\n"
        "y colocalo en alguna de estas rutas:\n  - " + "\n  - ".join(CANDIDATOS)
    )

print(f"Dataset encontrado: {path}")
df = spark.read.csv(path, header=True, schema=schema)

df.show(5)
df.printSchema()

In [ ]:
# ⚠️ CELDA OPTATIVA — NO EJECUTAR EN WINDOWS SIN winutils.exe
# Escribir Parquet en Windows necesita winutils.exe (HADOOP_HOME). Sin el, Spark falla con:
#   java.io.FileNotFoundException: HADOOP_HOME and hadoop.home.dir are unset.
# En Linux/macOS, o en Windows con winutils instalado, descomenta las 2 lineas siguientes.
# df.write.mode("overwrite").parquet("data_fraud.parquet")
# df = spark.read.parquet("data_fraud.parquet")

# Verificar nulos rápidamente (esta parte SI es segura en cualquier sistema)
df.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns]).show()

In [ ]:
# Añadir columnas de error de balance
df = df.withColumn("errorBalanceOrig", (F.col("oldbalanceOrg") - F.col("amount")) - F.col("newbalanceOrig"))
df = df.withColumn("errorBalanceDest", (F.col("oldbalanceDest") + F.col("amount")) - F.col("newbalanceDest"))

# Agrupar datos por usuario y calcular el promedio de transacciones previas
windowSpec = Window.partitionBy("nameOrig").orderBy("step").rowsBetween(Window.unboundedPreceding, -1)

df = df.withColumn("avg_prev_amount", F.avg("amount").over(windowSpec))

# Rellenar nulos de la primera transacción
df = df.na.fill({"avg_prev_amount": 0})

df.select("nameOrig", "amount", "avg_prev_amount", "isFraud").show(10)

In [ ]:
from pyspark.ml.feature import StringIndexer, VectorAssembler

# Convertir cada tipo de transacción (CASH_OUT, TRANSFER...) a números
indexer = StringIndexer(inputCol="type", outputCol="typeIndexed")
df_indexed = indexer.fit(df).transform(df)

feature_cols = [
    "step", "typeIndexed", "amount", 
    "oldbalanceOrg", "newbalanceOrig", 
    "oldbalanceDest", "newbalanceDest", 
    "errorBalanceOrig", "errorBalanceDest",
    "avg_prev_amount"
] #Columnas que aportan al modelo

assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
final_data = assembler.transform(df_indexed).select("features", "isFraud","amount")

final_data.show(5)

In [ ]:
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator

# Dividir datos en Entrenamiento
train_data, test_data = final_data.randomSplit([0.7, 0.3], seed=42)

# Entrenar el modelo
rf = RandomForestClassifier(labelCol="isFraud", featuresCol="features", numTrees=20)
model = rf.fit(train_data)

# Hacer predicciones
predictions = model.transform(test_data)

# Evaluar con el Área bajo la curva
evaluator = BinaryClassificationEvaluator(labelCol="isFraud", metricName="areaUnderROC")
auc = evaluator.evaluate(predictions)

# --- Metricas sobre la CLASE FRAUDE ---------------------------------------
# El dataset esta muy desbalanceado (~0.13% de fraude): el AUC sola enganya,
# porque predecir "todo legitimo" ya da un accuracy del 99.87%.
# Recall y precision se calculan sobre la clase minoritaria, que es la que importa.
cm = {(r["isFraud"], int(r["prediction"])): r["count"]
      for r in predictions.groupBy("isFraud", "prediction").count().collect()}
tp = cm.get((1, 1), 0); fn = cm.get((1, 0), 0)
fp = cm.get((0, 1), 0); tn = cm.get((0, 0), 0)

recall    = tp / (tp + fn) if (tp + fn) else 0.0
precision = tp / (tp + fp) if (tp + fp) else 0.0
f1        = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0

print(f"Fraudes reales en test : {tp + fn:,}")
print(f"Fraudes detectados     : {tp:,}  (falsos negativos: {fn:,})")
print(f"Falsas alarmas         : {fp:,}")
print(f"AUC-ROC   : {auc:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"Precision : {precision:.4f}")
print(f"F1        : {f1:.4f}")
predictions.select("isFraud", "prediction", "probability").show(10)

In [ ]:
predictions.filter(F.col("isFraud") == 0).select("isFraud", "prediction", "probability").show(10)

In [ ]:
falsos_negativos = predictions.filter((F.col("isFraud") == 1) & (F.col("prediction") == 0))
falsos_negativos.select("isFraud", "prediction", "probability").show(10)

In [ ]:
falsos_positivos = predictions.filter((F.col("isFraud") == 0) & (F.col("prediction") == 1))
falsos_positivos.select("isFraud", "prediction", "probability").show(10)

In [ ]:
predictions.groupBy("isFraud", "prediction").count().show() #Matriz de confusión

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Extraer importancias del modelo
importances = model.featureImportances
feature_list = feature_cols
feature_imp = pd.DataFrame(list(zip(feature_list, importances.toArray())), 
                           columns=['Característica', 'Importancia']).sort_values('Importancia', ascending=False)

plt.figure(figsize=(10,6))
sns.barplot(x='Importancia', y='Característica', data=feature_imp, palette='viridis')
plt.title('¿Qué variables delataron el fraude?')
plt.show()

In [ ]:
# Impacto economico
riesgo_fuga_res = predictions.filter((F.col("isFraud") == 1) & (F.col("prediction") == 0)).agg(F.sum("amount")).collect()[0][0]
fraude_detenido_res = predictions.filter((F.col("isFraud") == 1) & (F.col("prediction") == 1)).agg(F.sum("amount")).collect()[0][0]

riesgo_fuga = riesgo_fuga_res if riesgo_fuga_res is not None else 0
fraude_detenido = fraude_detenido_res if fraude_detenido_res is not None else 0

print(f"💰 Dinero SALVADO por el modelo: ${fraude_detenido:,.2f}")
print(f"💸 Dinero PERDIDO (Falsos Negativos): ${riesgo_fuga:,.2f}")


In [ ]:
# Métricas finales
total_registros = predictions.count()
auc_valor = auc 
denominador = fraude_detenido + riesgo_fuga
recuperacion_porcentaje = (fraude_detenido / denominador * 100) if denominador > 0 else 0

# Crear el resumen en un DataFrame de Pandas (más estable para reportes)
data_resumen = {
    "Métrica": [
        "📊 Registros Procesados", 
        "📈 AUC-ROC", 
        "🎯 Recall (Sensibilidad)", 
        "💰 Ahorro Estimado", 
        "🛡️ Eficiencia por Monto"
    ],
    "Valor": [
        f"{total_registros:,}", 
        f"{auc_valor:.4f}", 
        f"{recall*100:.2f}%",
        f"${fraude_detenido:,.2f}", 
        f"{recuperacion_porcentaje:.2f}%"
    ],
    "Interpretación": [
        "Volumen total de transacciones analizadas",
        "Capacidad de distinguir entre fraude y normal",
        "Porcentaje de casos de fraude capturados",
        "Impacto económico positivo directo",
        "Proporción del capital total protegido"
    ]
}

df_informe_pd = pd.DataFrame(data_resumen)

# Esto mostrará una tabla elegante en Jupyter
display(df_informe_pd)


In [ ]:
# Cierra la sesión y libera la memoria RAM y núcleos de CPU
spark.stop()